# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 4 — Attribution quality and negative controls

This notebook evaluates whether the atom-level attributions produced by a trained model
agree with established mutagenicity toxicophores more than chance, and whether that
agreement is a consequence of training.

Three design choices govern the comparison. First, all attribution methods are scored on
the same model-independent set of molecules — every toxicophore-bearing test positive — so
that lifts from different methods, and from trained and untrained networks, are directly
comparable. Second, each attribution method is applied to a single representative model of
its family, so that no method benefits from ensemble averaging that another does not
receive. Third, every arm of the comparison has its own negative control: the graph arm is
compared against a distribution of untrained initialisations and against a label-permuted
network, and the tree arm against a label-permuted model and against random attributions
passed through the same bit-to-atom mapping.

**Contents**

1. Environment and shared definitions
2. Data, models and the evaluation set
3. Toxicophore agreement
4. Attributions from the trained models
5. Ranking convention
6. Attribution AUROC and fidelity
7. Untrained null distribution
8. Trained ensemble and the paired comparison
9. Label-permuted controls
10. Network-free and mapping-only baselines
11. Explanation stability
12. Summary statistics

*Approximate runtime: 6-8 hours on a T4. Progress is checkpointed and the notebook resumes
from the last completed item.*

## 1. Environment and shared definitions

In [1]:
!pip install -q rdkit
!pip install -q xgboost
!pip install -q shap
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 60.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK    = "/content/drive/MyDrive/ames_pipeline"
DATADIR = f"{WORK}/data"
OUTDIR  = f"{WORK}/results"
CKPT    = f"{WORK}/ckpt"
FIGDIR  = f"{WORK}/figures"
MODELS  = f"{WORK}/models"
EXPL    = f"{WORK}/explain"
for d in (WORK, DATADIR, OUTDIR, CKPT, FIGDIR, MODELS, EXPL):
    os.makedirs(d, exist_ok=True)
print("Working directory:", WORK)

Mounted at /content/drive
Working directory: /content/drive/MyDrive/ames_pipeline


The module written in Parts 2 and 3 is reproduced here so that this notebook runs
independently of the others.

In [3]:
%%writefile /content/ames_lib.py
"""Shared definitions for the Ames mutagenicity pipeline."""

import random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try:
            return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception:
            return smi


def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs  = list(groups.values())
    rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d)
    n_train, n_train_valid = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr) + len(g) <= n_train:
            tr += g
        elif len(tr) + len(va) + len(g) <= n_train_valid:
            va += g
        else:
            te += g
    return d.loc[tr], d.loc[va], d.loc[te]


def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr


def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None:
            X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)


def one_hot(x, allowable):
    if x not in allowable:
        x = allowable[-1]
    return [int(x == a) for a in allowable]


def atom_features(atom):
    f  = one_hot(atom.GetSymbol(), ATOM_LIST)
    f += one_hot(str(atom.GetHybridization()), HYBRIDIZATION)
    f += [atom.GetDegree(), atom.GetFormalCharge(), int(atom.GetIsAromatic()),
          atom.GetTotalNumHs(), int(atom.IsInRing())]
    return f


def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(bond.GetIsConjugated()), int(bond.IsInRing())]


def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        ei += [[i, j], [j, i]]
        ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)


def build_dataset(d):
    graphs = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [g for g in graphs if g is not None]


class GNN(nn.Module):
    """Message-passing classifier. GINE and GAT consume bond features;
    GCN operates on connectivity alone."""

    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn":
                conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else:
                raise ValueError(f"Unknown architecture: {kind}")
            self.convs.append(conv)
            self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, data):
        x, edge_index, edge_attr, batch = (data.x, data.edge_index,
                                           data.edge_attr, data.batch)
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = (conv(x, edge_index, edge_attr) if self.use_edge_attr
                 else conv(x, edge_index))
            h = F.relu(bn(h))
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, batch), global_add_pool(x, batch)], dim=1)
        return self.head(g).squeeze(-1)


def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4,
              use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads,
               use_edge_attr=use_edge_attr).to(DEVICE)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps, logits = [], [], []
    for d in loader:
        d = d.to(DEVICE)
        logit = model(d)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(logit).cpu().numpy())
        logits.append(logit.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(logits)


def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, trial=None, use_edge_attr=True):
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    optimiser = torch.optim.Adam(model.parameters(), lr=params["lr"],
                                 weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    train_loader = DataLoader(tr_set, batch_size=params["batch_size"],
                              shuffle=True, drop_last=True)
    valid_loader = DataLoader(va_set, batch_size=256)
    best_auc, best_state, waited = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for d in train_loader:
            d = d.to(DEVICE)
            optimiser.zero_grad()
            criterion(model(d), d.y.view(-1)).backward()
            optimiser.step()
        y_va, p_va, _ = evaluate(model, valid_loader)
        auc = roc_auc_score(y_va, p_va) if len(np.unique(y_va)) > 1 else 0.5
        if auc > best_auc:
            best_auc, waited = auc, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_auc


class TensorView:
    """Lightweight stand-in for a Data object, used when x must carry gradients."""
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = (
            x, edge_index, edge_attr, batch)

Writing /content/ames_lib.py


In [4]:
import sys, json, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames_lib import *
from torch_geometric.loader import DataLoader

print("Device:", DEVICE, "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

SEED               = 42
PRIMARY_SPLIT_SEED = 101
MAX_EPOCHS, PATIENCE = 250, 30

N_UNTRAINED   = 500    # random initialisations forming the null distribution
N_TRAINED     = 50     # independently trained models forming the trained arm
N_IG_STEPS    = 32
K_FRAC        = 0.30   # fraction of atoms retained when scoring agreement
N_BOOTSTRAP   = 10000
EXPLAIN_SEED  = 42     # representative model used for the attribution comparison

HP_GNN     = json.load(open(f"{OUTDIR}/hpo_gnn.json"))
HP_CLASSIC = json.load(open(f"{OUTDIR}/hpo_classic.json"))

Device: cuda | Tesla T4


## 2. Data, models and the evaluation set

The primary partition is rebuilt, a representative graph model and a representative tree
model are fitted, and the evaluation set is defined as every test positive that contains at
least one toxicophore. This definition does not depend on any model, so the same molecules
are used for the trained networks, the untrained networks and the tree model.

In [5]:
from xgboost import XGBClassifier

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}")
                for f in ["train.csv", "valid.csv", "test.csv"]],
               ignore_index=True)[["Drug_ID", "smiles", "label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)

df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]

X_train, df_train = build_X(df_train); y_train = df_train["label"].values
X_valid, df_valid = build_X(df_valid); y_valid = df_valid["label"].values
X_test,  df_test  = build_X(df_test);  y_test  = df_test["label"].values

train_set, valid_set, test_set = (build_dataset(df_train), build_dataset(df_valid),
                                  build_dataset(df_test))
valid_loader = DataLoader(valid_set, batch_size=256)
test_loader  = DataLoader(test_set,  batch_size=256)

pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
                          dtype=torch.float, device=DEVICE)

# Representative tree model: the configuration selected in Part 1, single seed.
xgb_rep = XGBClassifier(n_estimators=1500, **HP_CLASSIC["XGBoost"],
                        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                        eval_metric="auc", early_stopping_rounds=50, tree_method="hist",
                        random_state=EXPLAIN_SEED, n_jobs=-1)
xgb_rep.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
p_xgb = xgb_rep.predict_proba(X_test)[:, 1]

# Representative graph model: reload the weights saved in Part 2 if available.
rep_path = f"{MODELS}/gin_seed{EXPLAIN_SEED}.pt"
gin_rep = build_gnn("gin", HP_GNN["gin"]["hidden"], HP_GNN["gin"]["n_layers"],
                    HP_GNN["gin"]["dropout"], HP_GNN["gin"].get("residual", True))
if os.path.exists(rep_path):
    gin_rep.load_state_dict(torch.load(rep_path, map_location=DEVICE)["state_dict"])
    print("Representative GIN loaded from Part 2")
else:
    gin_rep, _ = train_gnn("gin", HP_GNN["gin"], EXPLAIN_SEED, train_set, valid_set,
                           pos_weight, MAX_EPOCHS, PATIENCE)
    torch.save({"state_dict": gin_rep.state_dict()}, rep_path)
    print("Representative GIN trained and saved")
gin_rep.eval()
_, p_gin, _ = evaluate(gin_rep, test_loader)

from sklearn.metrics import roc_auc_score
print(f"Representative models: XGBoost ROC-AUC {roc_auc_score(y_test, p_xgb):.4f}, "
      f"GIN ROC-AUC {roc_auc_score(y_test, p_gin):.4f}")

Representative GIN loaded from Part 2
Representative models: XGBoost ROC-AUC 0.7833, GIN ROC-AUC 0.7997


## 3. Toxicophore agreement

Agreement is measured as the fraction of the highest-attributed 30% of atoms that belong to
a toxicophore match, divided by the fraction of toxicophore atoms in the molecule. A value
of one indicates agreement no better than selecting atoms at random.

Two details matter for reproducibility. The reported lift is the ratio of the mean observed
overlap to the mean chance overlap across molecules, not the mean of per-molecule ratios.
Ties in the attribution scores are broken at random rather than by atom index, since some
attribution schemes assign identical scores to many atoms and index order is not neutral.

In [6]:
TOXICOPHORES = {
    "aromatic_nitro":         "[a][N+](=O)[O-]",
    "aromatic_amine":         "[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
    "aromatic_nitroso":       "[a][NX2]=[OX1]",
    "alkyl_nitrite":          "[CX4][OX2][NX2]=[OX1]",
    "nitrosamine":            "[NX3][NX2]=[OX1]",
    "epoxide":                "C1OC1",
    "aziridine":              "C1NC1",
    "azide":                  "[NX2]=[NX2+]=[NX1-]",
    "azo_aromatic":           "[a][NX2]=[NX2][a]",
    "aliphatic_halide":       "[CX4][F,Cl,Br,I]",
    "hetero_hetero_bond":     "[N,O;H1,H2][N,O;H1,H2]",
    "polycyclic_aromatic":    "c1ccc2ccccc2c1",
    "aromatic_hydroxylamine": "[a][NX3;H1][OX2;H1]",
    "michael_acceptor":       "[CX3]=[CX3][CX3]=[OX1]",
    "aromatic_azoxy":         "[a][NX2]=[NX3+]([O-])[a]",
    "nitrogen_mustard":       "[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
    "aromatic_hydrazine":     "[a][NX3;H1][NX3;H2]",
    "alkyl_sulfonate":        "[CX4][OX2][SX4](=[OX1])(=[OX1])",
}
TOX_PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in TOXICOPHORES.items()}
assert all(p is not None for p in TOX_PATTERNS.values())

def toxicophore_atoms(mol):
    atoms = set()
    for patt in TOX_PATTERNS.values():
        for match in mol.GetSubstructMatches(patt):
            atoms.update(match)
    return atoms

# Model-independent evaluation set.
EVAL = []
for pos, (smi, lab) in enumerate(zip(df_test["canonical"], y_test)):
    if lab != 1:
        continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0:
        continue
    tox = toxicophore_atoms(mol)
    if not tox or len(tox) == mol.GetNumAtoms():
        continue
    EVAL.append({"row": pos, "smiles": smi, "mol": mol, "tox": tox,
                 "n_atoms": mol.GetNumAtoms(),
                 "chance": len(tox) / mol.GetNumAtoms()})

EVAL_GRAPHS = [smiles_to_data(e["smiles"], 1.0) for e in EVAL]
CHANCE  = np.array([e["chance"] for e in EVAL])
N_EVAL  = len(EVAL)

def overlap_from_scores(scores_list, k_frac=K_FRAC, tie_seed=0):
    """Per-molecule overlap between the top-k attributed atoms and toxicophore atoms."""
    rng = np.random.default_rng(tie_seed)
    out = np.full(N_EVAL, np.nan)
    for i, (e, sc) in enumerate(zip(EVAL, scores_list)):
        if sc is None or len(sc) != e["n_atoms"]:
            continue
        n = e["n_atoms"]
        k = max(1, int(round(k_frac * n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"]) / k
    return out

def lift_from_overlap(ov, chance=None):
    chance = CHANCE if chance is None else chance
    m = ~np.isnan(ov)
    return float(ov[m].mean() / chance[m].mean()) if m.sum() else np.nan

# The attainable maximum is not 1 / mean(chance). When a molecule contains fewer
# toxicophore atoms than the top-k selection retains, even a perfect ranking must
# fill the remaining positions with atoms outside any toxicophore. The attainable
# ceiling is therefore measured directly, from a ranking that places every
# toxicophore atom first.
ORACLE = [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])])
          for e in EVAL]
CEILING            = lift_from_overlap(overlap_from_scores(ORACLE, tie_seed=0))
CEILING_THEORETIC  = 1.0 / CHANCE.mean()

print(f"Evaluation set: {N_EVAL} toxicophore-bearing test positives")
print(f"Mean chance overlap        : {CHANCE.mean():.4f}")
print(f"Attainable maximum lift    : {CEILING:.4f}")
print(f"Theoretical upper bound    : {CEILING_THEORETIC:.4f} "
      f"(reached only if every molecule were at least {100*K_FRAC:.0f}% toxicophore)")

Evaluation set: 237 toxicophore-bearing test positives
Mean chance overlap        : 0.4360
Attainable maximum lift    : 1.9144
Theoretical upper bound    : 2.2934 (reached only if every molecule were at least 30% toxicophore)


In [7]:
# Verification of the scoring machinery before it is applied to any model.
_random_lifts = [lift_from_overlap(overlap_from_scores(
                    [np.random.default_rng(90_000 + i * 997 + k).random(e["n_atoms"])
                     for k, e in enumerate(EVAL)], tie_seed=90_000 + i))
                 for i in range(100)]
_tied_lift = np.mean([lift_from_overlap(overlap_from_scores(
                        [np.ones(e["n_atoms"]) for e in EVAL], tie_seed=i))
                      for i in range(100)])
_oracle_lift = lift_from_overlap(overlap_from_scores(ORACLE, tie_seed=SEED))
_anti = [-o for o in ORACLE]
_anti_lift = lift_from_overlap(overlap_from_scores(_anti, tie_seed=SEED))

print(f"Random atom ranking       : {np.mean(_random_lifts):.4f}   (expected 1.00)")
print(f"Uniform scores, ties only : {_tied_lift:.4f}   (expected 1.00)")
print(f"Oracle ranking            : {_oracle_lift:.4f}   (attainable maximum {CEILING:.4f})")
print(f"Inverted oracle ranking   : {_anti_lift:.4f}   (expected well below 1.00)")

assert abs(np.mean(_random_lifts) - 1) < 0.03, "a random ranking should not beat chance"
assert abs(_tied_lift - 1) < 0.05, "tie-breaking is not neutral"
assert abs(_oracle_lift - CEILING) < 0.02, "oracle ranking should reach the attainable maximum"
assert _anti_lift < 1.0, "inverting the oracle should fall below chance"
assert CEILING > 1.2, "the metric has too little headroom to be informative"

Random atom ranking       : 1.0011   (expected 1.00)
Uniform scores, ties only : 0.9982   (expected 1.00)
Oracle ranking            : 1.9144   (attainable maximum 1.9144)
Inverted oracle ranking   : 0.2125   (expected well below 1.00)


## 4. Attributions from the trained models

Integrated Gradients is computed over a 32-step path from a zero baseline, in batches, so
that the same routine can be applied several hundred times when the null distribution is
constructed. Saliency and Gradient x Input are computed alongside it, which distinguishes
behaviour specific to Integrated Gradients from behaviour common to gradient attribution.

TreeSHAP values are computed for the fingerprint bits and distributed to atoms through the
environments each bit encodes, dividing each bit's contribution equally among its
environments and among the atoms in each environment.

In [8]:
def batched_gradient_scores(model, graphs, method="ig", n_steps=N_IG_STEPS,
                            batch_size=64, signed=False):
    """Atom scores from a gradient attribution method, evaluated in batches.

    Batching does not change per-molecule results: the graphs in a batch are
    disconnected, pooling is per-graph, and the model is in evaluation mode so
    batch normalisation uses fixed statistics.
    """
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        batch = batch.to(DEVICE)
        x0 = batch.x.clone()
        if method == "ig":
            total = torch.zeros_like(x0)
            for alpha in torch.linspace(1.0 / n_steps, 1.0, n_steps):
                xa = (x0 * alpha).detach().requires_grad_(True)
                logit = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
                total += torch.autograd.grad(logit.sum(), xa)[0]
            attr = x0 * (total / n_steps)
        else:
            xa = x0.clone().detach().requires_grad_(True)
            logit = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            grad = torch.autograd.grad(logit.sum(), xa)[0]
            attr = grad if method == "saliency" else x0 * grad
        scores = (attr.sum(dim=1) if signed else attr.abs().sum(dim=1))
        scores = scores.detach().cpu().numpy()
        b = batch.batch.cpu().numpy()
        for g in range(batch.num_graphs):
            out.append(scores[b == g])
    return out

import shap
from rdkit.Chem import rdFingerprintGenerator as rfg

def fp_with_bitinfo(mol):
    ao = rfg.AdditionalOutput(); ao.AllocateBitInfoMap()
    fp = morgan_gen.GetFingerprint(mol, additionalOutput=ao)
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr, ao.GetBitInfoMap()

def bit_to_atoms(mol, atom_idx, radius):
    if radius == 0:
        return {atom_idx}
    env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
    atoms = {atom_idx}
    for bidx in env:
        bond = mol.GetBondWithIdx(bidx)
        atoms.add(bond.GetBeginAtomIdx()); atoms.add(bond.GetEndAtomIdx())
    return atoms

BITINFO = []
for e in EVAL:
    try:
        _, bi = fp_with_bitinfo(e["mol"])
    except Exception:
        bi = {}
    BITINFO.append(bi)

def shap_to_atom_scores(shap_values, signed=False):
    """Distribute bit-level values to atoms through the environments each bit encodes.

    Accepts either a matrix with one row per evaluation molecule, or a single
    vector applied to every molecule (used for the mapping-only control).
    """
    sv = np.asarray(shap_values)
    per_molecule = sv.ndim == 2
    if per_molecule and sv.shape[0] != N_EVAL:
        raise ValueError(f"expected {N_EVAL} rows, received {sv.shape[0]}")
    if sv.shape[-1] != FP_BITS:
        raise ValueError(f"expected {FP_BITS} columns, received {sv.shape[-1]}")
    out = []
    for i, (e, bi) in enumerate(zip(EVAL, BITINFO)):
        vec = sv[i] if per_molecule else sv
        scores = np.zeros(e["n_atoms"], dtype=float)
        for bit, envs in bi.items():
            v = float(vec[bit])
            if v == 0.0 or len(envs) == 0:
                continue
            for atom_idx, radius in envs:
                atoms = bit_to_atoms(e["mol"], atom_idx, radius)
                share = v / (len(envs) * len(atoms))
                for a in atoms:
                    scores[a] += share
        out.append(scores if signed else np.abs(scores))
    return out

def tree_shap_matrix(model, X):
    """SHAP values for the positive class as an (n_samples, n_bits) array."""
    values = shap.TreeExplainer(model).shap_values(X)
    if isinstance(values, list):
        values = values[-1]
    values = np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, -1]
    return values

X_eval   = np.array([fp_array(e["smiles"]) for e in EVAL])
SHAP_EVAL = tree_shap_matrix(xgb_rep, X_eval)
assert SHAP_EVAL.shape == (N_EVAL, FP_BITS), SHAP_EVAL.shape
print("SHAP values:", SHAP_EVAL.shape)

SHAP values: (237, 2048)


## 5. Ranking convention

Gradient attributions are commonly aggregated as an absolute sum over feature dimensions,
while SHAP values are commonly read as signed contributions towards the positive class.
These are different definitions of "most attributed", so both conventions are evaluated for
both methods and the resulting lifts are reported together.

In [9]:
rows = []
attr_cache = {}

for method in ["ig", "gradinput", "saliency"]:
    for signed in [False, True]:
        sc = batched_gradient_scores(gin_rep, EVAL_GRAPHS, method=method, signed=signed)
        ov = overlap_from_scores(sc, tie_seed=SEED)
        key = f"GIN_{method}_{'signed' if signed else 'abs'}"
        attr_cache[key] = (sc, ov)
        rows.append({"arm": "graph", "method": method,
                     "convention": "signed" if signed else "absolute",
                     "n": int((~np.isnan(ov)).sum()), "lift": lift_from_overlap(ov)})

for signed in [False, True]:
    sc = shap_to_atom_scores(SHAP_EVAL, signed=signed)
    ov = overlap_from_scores(sc, tie_seed=SEED)
    key = f"XGB_treeshap_{'signed' if signed else 'abs'}"
    attr_cache[key] = (sc, ov)
    rows.append({"arm": "tree", "method": "treeshap",
                 "convention": "signed" if signed else "absolute",
                 "n": int((~np.isnan(ov)).sum()), "lift": lift_from_overlap(ov)})

conv_df = pd.DataFrame(rows)
conv_df["pct_of_ceiling"] = 100 * (conv_df["lift"] - 1) / (CEILING - 1)
conv_df.to_csv(f"{OUTDIR}/table10_ranking_convention.csv", index=False)
print(conv_df.round(4).to_string(index=False))

  arm    method convention   n   lift  pct_of_ceiling
graph        ig   absolute 237 1.2265         24.7664
graph        ig     signed 237 1.0679          7.4213
graph gradinput   absolute 237 1.2164         23.6629
graph gradinput     signed 237 1.1702         18.6095
graph  saliency   absolute 237 1.1784         19.5073
graph  saliency     signed 237 1.0916         10.0161
 tree  treeshap   absolute 237 1.4416         48.2902
 tree  treeshap     signed 237 1.5422         59.2899


## 6. GNNExplainer, attribution AUROC and fidelity

GNNExplainer is applied to the same representative model and the same molecules. In
addition to the threshold-dependent overlap, a threshold-free attribution AUROC treats the
atom scores as a ranking of toxicophore membership. Fidelity measures how far the predicted
probability moves when the most-attributed atoms are removed and when only they are
retained.

In [10]:
from torch_geometric.explain import Explainer, GNNExplainer
import torch.nn as nn

class ExplainWrapper(nn.Module):
    """Tensor-argument interface required by the Explainer API."""
    def __init__(self, model):
        super().__init__(); self.model = model
    def forward(self, x, edge_index, edge_attr=None, batch=None, **kwargs):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        return self.model(TensorView(x, edge_index, edge_attr, batch)).view(-1)

def gnnexplainer_scores(model, graphs, epochs=100, seed=SEED):
    set_seed(seed)
    wrapper = ExplainWrapper(model).to(DEVICE).eval()
    explainer = Explainer(
        model=wrapper, algorithm=GNNExplainer(epochs=epochs),
        explanation_type="model", node_mask_type="attributes", edge_mask_type="object",
        model_config=dict(mode="binary_classification", task_level="graph",
                          return_type="raw"))
    out, failures = [], 0
    for g in graphs:
        g = g.to(DEVICE)
        batch = torch.zeros(g.x.size(0), dtype=torch.long, device=DEVICE)
        try:
            ex = explainer(g.x, g.edge_index, edge_attr=g.edge_attr, batch=batch)
            out.append(ex.node_mask.abs().sum(dim=1).detach().cpu().numpy())
        except Exception as err:
            failures += 1
            if failures == 1:
                print("  first failure:", type(err).__name__, err)
            out.append(None)
    print(f"  GNNExplainer produced attributions for "
          f"{len(graphs) - failures}/{len(graphs)} molecules")
    return out

gnnex_path = f"{EXPL}/gnnexplainer_scores.npy"
if os.path.exists(gnnex_path):
    gnnex_sc = list(np.load(gnnex_path, allow_pickle=True))
    print("GNNExplainer scores loaded from checkpoint")
else:
    t0 = time.time()
    gnnex_sc = gnnexplainer_scores(gin_rep, EVAL_GRAPHS)
    n_ok = sum(s is not None for s in gnnex_sc)
    if n_ok < 0.5 * N_EVAL:
        raise RuntimeError(
            f"GNNExplainer succeeded on only {n_ok}/{N_EVAL} molecules; "
            "the explanation comparison would not be meaningful.")
    np.save(gnnex_path, np.array(gnnex_sc, dtype=object))
    print(f"GNNExplainer completed in {(time.time()-t0)/60:.1f} min")

gnnex_ov = overlap_from_scores(gnnex_sc, tie_seed=SEED)
attr_cache["GIN_gnnexplainer_abs"] = (gnnex_sc, gnnex_ov)
print(f"GNNExplainer lift: {lift_from_overlap(gnnex_ov):.4f}")

  GNNExplainer produced attributions for 237/237 molecules
GNNExplainer completed in 3.3 min
GNNExplainer lift: 0.8978


In [11]:
def attribution_auroc(scores_list):
    aucs = []
    for e, sc in zip(EVAL, scores_list):
        if sc is None or len(sc) != e["n_atoms"]:
            continue
        y = np.zeros(e["n_atoms"], dtype=int)
        y[list(e["tox"])] = 1
        if len(np.unique(y)) < 2:
            continue
        aucs.append(roc_auc_score(y, np.asarray(sc, dtype=float)))
    return np.array(aucs)

@torch.no_grad()
def masked_probability(model, graph, keep_mask):
    g = graph.clone().to(DEVICE)
    g.x = g.x.clone()
    g.x[~torch.as_tensor(keep_mask, device=DEVICE)] = 0.0
    batch = torch.zeros(g.x.size(0), dtype=torch.long, device=DEVICE)
    return torch.sigmoid(model(TensorView(g.x, g.edge_index, g.edge_attr, batch))).item()

def fidelity(model, graphs, scores_list, k_frac=K_FRAC):
    drop, keep = [], []
    for g, sc in zip(graphs, scores_list):
        if sc is None or len(sc) != g.x.size(0):
            continue
        n = g.x.size(0); k = max(1, int(round(k_frac * n)))
        top = np.argsort(-np.asarray(sc, dtype=float))[:k]
        base = masked_probability(model, g, np.ones(n, dtype=bool))
        m_drop = np.ones(n, dtype=bool); m_drop[top] = False
        m_keep = np.zeros(n, dtype=bool); m_keep[top] = True
        drop.append(base - masked_probability(model, g, m_drop))
        keep.append(base - masked_probability(model, g, m_keep))
    fp, fm = float(np.mean(drop)), float(np.mean(keep))
    char = 2 * fp * (1 - fm) / (fp + (1 - fm)) if (fp + (1 - fm)) != 0 else np.nan
    return fp, fm, char

rows = []
for label, key in [("XGBoost (TreeSHAP, signed)", "XGB_treeshap_signed"),
                   ("XGBoost (TreeSHAP, absolute)", "XGB_treeshap_abs"),
                   ("GIN (Integrated Gradients)", "GIN_ig_abs"),
                   ("GIN (Gradient x Input)", "GIN_gradinput_abs"),
                   ("GIN (Saliency)", "GIN_saliency_abs"),
                   ("GIN (GNNExplainer)", "GIN_gnnexplainer_abs")]:
    sc, ov = attr_cache[key]
    aucs = attribution_auroc(sc)
    row = {"explainer": label, "n": int((~np.isnan(ov)).sum()),
           "lift": lift_from_overlap(ov),
           "pct_of_ceiling": 100 * (lift_from_overlap(ov) - 1) / (CEILING - 1),
           "attribution_auroc": float(np.mean(aucs)), "n_auroc": len(aucs)}
    if key.startswith("GIN"):
        fp, fm, char = fidelity(gin_rep, EVAL_GRAPHS, sc)
        row.update({"fidelity_plus": fp, "fidelity_minus": fm, "characterisation": char})
    rows.append(row)

expl_df = pd.DataFrame(rows)
expl_df.to_csv(f"{OUTDIR}/table11_explainers.csv", index=False)
print(expl_df.round(4).to_string(index=False))

np.savez(f"{EXPL}/overlaps_trained.npz",
         chance=CHANCE, **{k: v[1] for k, v in attr_cache.items()})

                   explainer   n   lift  pct_of_ceiling  attribution_auroc  n_auroc  fidelity_plus  fidelity_minus  characterisation
  XGBoost (TreeSHAP, signed) 237 1.5422         59.2899             0.7979      237            NaN             NaN               NaN
XGBoost (TreeSHAP, absolute) 237 1.4416         48.2902             0.7397      237            NaN             NaN               NaN
  GIN (Integrated Gradients) 237 1.2265         24.7664             0.6150      237        -0.0536         -0.1451           -0.1125
      GIN (Gradient x Input) 237 1.2164         23.6629             0.6051      237        -0.0009         -0.1898           -0.0018
              GIN (Saliency) 237 1.1784         19.5073             0.6132      237         0.0014         -0.1963            0.0028
          GIN (GNNExplainer) 237 0.8978        -11.1744             0.4763      237         0.0544         -0.1881            0.1041


## 7. Untrained null distribution

Five hundred networks are drawn from the initialisation distribution of each architecture
and scored with the same attribution routine. This gives the distribution of toxicophore
agreement that the architecture and the attribution method produce before any training,
against which a trained model must be compared.

Percentiles and the standard deviation are reported as the primary summary. The observed
range is also recorded, but it grows with the number of draws and is therefore not a stable
description of the distribution.

In [12]:
null_path = f"{EXPL}/untrained_null.csv"
ov_path   = {a: f"{EXPL}/untrained_overlaps_{a}.npy" for a in ["gin", "gcn", "gat"]}

null_rows = pd.read_csv(null_path).to_dict("records") if os.path.exists(null_path) else []
done = {(r["arch"], r["init"]) for r in null_rows}
overlaps = {a: (list(np.load(ov_path[a])) if os.path.exists(ov_path[a]) else [])
            for a in ov_path}

for arch in ["gin", "gcn", "gat"]:
    hp = HP_GNN[arch]
    t0 = time.time()
    for i in range(N_UNTRAINED):
        if (arch, i) in done:
            continue
        set_seed(10_000 + i)
        m = build_gnn(arch, hp["hidden"], hp["n_layers"], hp["dropout"],
                      hp.get("residual", True))
        sc = batched_gradient_scores(m, EVAL_GRAPHS, method="ig")
        ov = overlap_from_scores(sc, tie_seed=10_000 + i)
        overlaps[arch].append(ov)
        null_rows.append({"arch": arch, "init": i, "lift": lift_from_overlap(ov),
                          "mean_overlap": float(np.nanmean(ov))})
        del m; torch.cuda.empty_cache()
        if (i + 1) % 50 == 0:
            pd.DataFrame(null_rows).to_csv(null_path, index=False)
            np.save(ov_path[arch], np.array(overlaps[arch]))
            print(f"  {arch} {i+1}/{N_UNTRAINED}  ({(time.time()-t0)/60:.1f} min)")
    pd.DataFrame(null_rows).to_csv(null_path, index=False)
    np.save(ov_path[arch], np.array(overlaps[arch]))

null_df = pd.read_csv(null_path)
summary = null_df.groupby("arch")["lift"].agg(
    n="count", mean="mean", sd="std",
    p2_5=lambda s: np.percentile(s, 2.5),
    p97_5=lambda s: np.percentile(s, 97.5),
    min="min", max="max").round(4)
summary.to_csv(f"{OUTDIR}/table12_untrained_null.csv")
print(summary.to_string())
print("\nThe percentiles and standard deviation describe the distribution; the")
print("observed range grows with the number of draws and is reported for reference only.")

  gin 50/500  (0.6 min)
  gin 100/500  (1.2 min)
  gin 150/500  (1.8 min)
  gin 200/500  (2.4 min)
  gin 250/500  (3.1 min)
  gin 300/500  (3.7 min)
  gin 350/500  (4.3 min)
  gin 400/500  (4.9 min)
  gin 450/500  (5.5 min)
  gin 500/500  (6.1 min)
  gcn 50/500  (0.6 min)
  gcn 100/500  (1.2 min)
  gcn 150/500  (1.8 min)
  gcn 200/500  (2.3 min)
  gcn 250/500  (2.9 min)
  gcn 300/500  (3.5 min)
  gcn 350/500  (4.1 min)
  gcn 400/500  (4.7 min)
  gcn 450/500  (5.3 min)
  gcn 500/500  (5.9 min)
  gat 50/500  (1.0 min)
  gat 100/500  (2.0 min)
  gat 150/500  (3.0 min)
  gat 200/500  (4.0 min)
  gat 250/500  (4.9 min)
  gat 300/500  (5.9 min)
  gat 350/500  (6.9 min)
  gat 400/500  (7.9 min)
  gat 450/500  (8.9 min)
  gat 500/500  (9.9 min)
        n    mean      sd    p2_5   p97_5     min     max
arch                                                     
gat   500  1.0859  0.1251  0.7976  1.2399  0.7251  1.2751
gcn   500  1.0370  0.1414  0.7580  1.2450  0.7012  1.2851
gin   500  1.1156  0.

## 8. Trained ensemble and the paired comparison

Fifty networks are trained from independent seeds and scored with the same routine. The
comparison against the untrained null is paired at the level of individual molecules: for
each molecule, the mean overlap across trained models is compared with the mean overlap
across untrained models.

Both the number of molecules that improve and the matched-pairs rank-biserial correlation
are reported. These answer different questions — how often training helps, and how
consistently it helps relative to the magnitude of the changes — and can point in different
directions.

In [13]:
trained_path = f"{EXPL}/trained_overlaps.npz"
if os.path.exists(trained_path):
    z = np.load(trained_path)
    trained_ov = z["overlaps"]; trained_lifts = z["lifts"]
    print(f"Trained arm loaded from checkpoint: {trained_ov.shape[0]} models")
else:
    trained_ov, trained_lifts = [], []

start = len(trained_lifts)
if start < N_TRAINED:
    trained_ov = list(trained_ov); trained_lifts = list(trained_lifts)
    t0 = time.time()
    for s in range(start, N_TRAINED):
        m, _ = train_gnn("gin", HP_GNN["gin"], 20_000 + s, train_set, valid_set,
                         pos_weight, MAX_EPOCHS, PATIENCE)
        m.eval()
        sc = batched_gradient_scores(m, EVAL_GRAPHS, method="ig")
        ov = overlap_from_scores(sc, tie_seed=20_000 + s)
        trained_ov.append(ov); trained_lifts.append(lift_from_overlap(ov))
        del m; torch.cuda.empty_cache()
        np.savez(trained_path, overlaps=np.array(trained_ov),
                 lifts=np.array(trained_lifts))
        print(f"  trained model {s+1}/{N_TRAINED}: lift {trained_lifts[-1]:.4f}  "
              f"({(time.time()-t0)/60:.1f} min elapsed)")
    trained_ov = np.array(trained_ov); trained_lifts = np.array(trained_lifts)

print(f"Trained arm: mean lift {trained_lifts.mean():.4f} "
      f"(sd {trained_lifts.std(ddof=1):.4f}, n = {len(trained_lifts)})")

  trained model 1/50: lift 1.2362  (1.4 min elapsed)
  trained model 2/50: lift 0.9377  (5.6 min elapsed)
  trained model 3/50: lift 1.1861  (8.2 min elapsed)
  trained model 4/50: lift 1.1746  (10.3 min elapsed)
  trained model 5/50: lift 1.2424  (12.6 min elapsed)
  trained model 6/50: lift 1.2580  (14.4 min elapsed)
  trained model 7/50: lift 1.2056  (17.1 min elapsed)
  trained model 8/50: lift 1.3550  (19.4 min elapsed)
  trained model 9/50: lift 1.2658  (21.5 min elapsed)
  trained model 10/50: lift 1.3379  (23.7 min elapsed)
  trained model 11/50: lift 1.2415  (26.4 min elapsed)
  trained model 12/50: lift 1.2933  (28.1 min elapsed)
  trained model 13/50: lift 1.3596  (30.8 min elapsed)
  trained model 14/50: lift 1.3136  (32.4 min elapsed)
  trained model 15/50: lift 1.2725  (34.5 min elapsed)
  trained model 16/50: lift 1.4324  (36.2 min elapsed)
  trained model 17/50: lift 1.2081  (38.2 min elapsed)
  trained model 18/50: lift 1.1892  (40.1 min elapsed)
  trained model 19/50:

In [14]:
from scipy.stats import wilcoxon, rankdata

untrained_ov = np.load(f"{EXPL}/untrained_overlaps_gin.npy")
assert untrained_ov.shape[1] == N_EVAL, untrained_ov.shape
print(f"Untrained arm: {untrained_ov.shape[0]} initialisations")

tr_mean = np.nanmean(trained_ov, axis=0)
un_mean = np.nanmean(untrained_ov, axis=0)
valid   = ~np.isnan(tr_mean) & ~np.isnan(un_mean)
d       = (tr_mean - un_mean)[valid]

# Two distinct quantities, reported separately.
#   aggregate lift : computed from the molecule-averaged overlaps; this is what the
#                    paired test and the bootstrap interval refer to.
#   mean model lift: the average of the per-model lifts, whose spread describes how
#                    much a single draw can differ from the mean.
lift_tr_agg = lift_from_overlap(tr_mean)
lift_un_agg = lift_from_overlap(un_mean)
null_gin    = null_df[null_df.arch == "gin"]["lift"].values
lift_tr_mod, sd_tr = trained_lifts.mean(), trained_lifts.std(ddof=1)
lift_un_mod, sd_un = null_gin.mean(), null_gin.std(ddof=1)

stat, p_w = wilcoxon(d)
nz = d[d != 0]; rk = rankdata(np.abs(nz))
W_pos, W_neg = rk[nz > 0].sum(), rk[nz < 0].sum()
rb = (W_pos - W_neg) / (W_pos + W_neg)

rng = np.random.default_rng(SEED)
tr_v, un_v, ch_v, n_v = tr_mean[valid], un_mean[valid], CHANCE[valid], int(valid.sum())
boot = np.empty(N_BOOTSTRAP)
for b in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_v, n_v)
    c = ch_v[idx].mean()
    boot[b] = tr_v[idx].mean() / c - un_v[idx].mean() / c

frac = ((rng.normal(lift_un_mod, sd_un / np.sqrt(len(null_gin)), 200_000) - 1) /
        (rng.normal(lift_tr_mod, sd_tr / np.sqrt(len(trained_lifts)), 200_000) - 1))
frac = frac[(frac > -1) & (frac < 2)]

result = {
    "n_molecules": n_v,
    "n_untrained": len(null_gin), "n_trained": len(trained_lifts),
    "untrained_lift_aggregate": lift_un_agg,
    "untrained_lift_mean_over_models": lift_un_mod, "untrained_lift_sd": sd_un,
    "untrained_p2_5": np.percentile(null_gin, 2.5),
    "untrained_p97_5": np.percentile(null_gin, 97.5),
    "trained_lift_aggregate": lift_tr_agg,
    "trained_lift_mean_over_models": lift_tr_mod, "trained_lift_sd": sd_tr,
    "lift_difference_aggregate": lift_tr_agg - lift_un_agg,
    "diff_ci_low": np.percentile(boot, 2.5), "diff_ci_high": np.percentile(boot, 97.5),
    "wilcoxon_p": p_w, "rank_biserial_kerby": rb,
    "n_improved": int((d > 0).sum()), "n_worsened": int((d < 0).sum()),
    "n_unchanged": int((d == 0).sum()), "pct_improved": 100 * (d > 0).mean(),
    "empirical_p": (1 + (null_gin >= lift_tr_mod).sum()) / (len(null_gin) + 1),
    "pretraining_fraction": (lift_un_mod - 1) / (lift_tr_mod - 1),
    "pretraining_frac_ci_low": np.percentile(frac, 2.5),
    "pretraining_frac_ci_high": np.percentile(frac, 97.5),
}
pd.DataFrame([result]).to_csv(f"{OUTDIR}/table13_training_gain.csv", index=False)
np.save(f"{EXPL}/paired_differences.npy", d)
for k, v in result.items():
    print(f"  {k:34s}: {v if isinstance(v, int) else round(float(v), 5)}")
print("\nn_improved counts how often training helps; the rank-biserial correlation")
print("weights each molecule by the size of its change. They answer different")
print("questions and are reported separately.")

Untrained arm: 500 initialisations
  n_molecules                       : 237
  n_untrained                       : 500
  n_trained                         : 50
  untrained_lift_aggregate          : 1.11564
  untrained_lift_mean_over_models   : 1.11564
  untrained_lift_sd                 : 0.05882
  untrained_p2_5                    : 0.98943
  untrained_p97_5                   : 1.22616
  trained_lift_aggregate            : 1.24204
  trained_lift_mean_over_models     : 1.24204
  trained_lift_sd                   : 0.09917
  lift_difference_aggregate         : 0.1264
  diff_ci_low                       : 0.05727
  diff_ci_high                      : 0.2023
  wilcoxon_p                        : 0.00128
  rank_biserial_kerby               : 0.24136
  n_improved                        : 126
  n_worsened                        : 111
  n_unchanged                       : 0
  pct_improved                      : 53.16456
  empirical_p                       : 0.02196
  pretraining_fraction     

## 9. Label-permuted controls

Both families are given a data-randomisation control: a model of the same architecture and
configuration is trained on permuted labels and its attributions are scored in the same
way. A network that memorises permuted labels has learned no relationship between structure
and mutagenicity, so any remaining toxicophore agreement is not evidence of learned
chemistry.

In [15]:
drt_path = f"{OUTDIR}/table14_label_permuted.csv"
if os.path.exists(drt_path):
    drt_df = pd.read_csv(drt_path)
else:
    rng = np.random.default_rng(SEED)
    rows = []

    # Graph arm
    perm = rng.permutation(len(train_set))
    train_perm = []
    for g, j in zip(train_set, perm):
        h = g.clone(); h.y = train_set[j].y.clone(); train_perm.append(h)
    m_perm, _ = train_gnn("gin", HP_GNN["gin"], SEED, train_perm, valid_set,
                          pos_weight, MAX_EPOCHS, PATIENCE)
    m_perm.eval()
    y_tr = np.array([float(g.y.item()) for g in train_perm])
    _, p_tr, _ = evaluate(m_perm, DataLoader(train_perm, batch_size=256))
    _, p_te, _ = evaluate(m_perm, test_loader)
    sc = batched_gradient_scores(m_perm, EVAL_GRAPHS, method="ig")
    ov = overlap_from_scores(sc, tie_seed=SEED)
    rows.append({"arm": "graph", "model": "GIN (permuted labels)",
                 "train_auc": roc_auc_score(y_tr, p_tr),
                 "test_auc": roc_auc_score(y_test, p_te),
                 "pred_sd": float(np.std(p_te)),
                 "lift": lift_from_overlap(ov), "n": int((~np.isnan(ov)).sum())})
    del m_perm; torch.cuda.empty_cache()

    # Tree arm
    y_perm = rng.permutation(y_train)
    xgb_perm = XGBClassifier(n_estimators=1500, **HP_CLASSIC["XGBoost"],
                             eval_metric="auc", early_stopping_rounds=50,
                             tree_method="hist", random_state=SEED, n_jobs=-1)
    xgb_perm.fit(X_train, y_perm, eval_set=[(X_valid, y_valid)], verbose=False)
    sv = shap.TreeExplainer(xgb_perm).shap_values(X_eval)
    ov = overlap_from_scores(shap_to_atom_scores(sv, signed=True), tie_seed=SEED)
    rows.append({"arm": "tree", "model": "XGBoost (permuted labels)",
                 "train_auc": roc_auc_score(y_perm, xgb_perm.predict_proba(X_train)[:, 1]),
                 "test_auc": roc_auc_score(y_test, xgb_perm.predict_proba(X_test)[:, 1]),
                 "pred_sd": float(np.std(xgb_perm.predict_proba(X_test)[:, 1])),
                 "lift": lift_from_overlap(ov), "n": int((~np.isnan(ov)).sum())})

    drt_df = pd.DataFrame(rows)
    drt_df.to_csv(drt_path, index=False)
print(drt_df.round(4).to_string(index=False))

  arm                     model  train_auc  test_auc  pred_sd   lift   n
graph     GIN (permuted labels)     0.5134    0.6582   0.0049 1.1440 237
 tree XGBoost (permuted labels)     0.6376    0.5010   0.0461 0.9458 237


## 10. Network-free and mapping-only baselines

Two further controls isolate the contribution of the machinery that surrounds the model.
The first ranks atoms by a random weighting of their own feature vectors, with no network
and no message passing at all. The second passes random values through the bit-to-atom
mapping used for the tree attributions. Either baseline exceeding one would indicate that
part of the observed agreement arises before any model is consulted.

In [16]:
base_path = f"{OUTDIR}/table15_baselines.csv"
if os.path.exists(base_path):
    base_df = pd.read_csv(base_path)
else:
    rng = np.random.default_rng(SEED)
    X_atoms = [np.array([atom_features(a) for a in e["mol"].GetAtoms()], dtype=float)
               for e in EVAL]
    rows = []

    lifts = []
    for i in range(200):
        g = np.abs(rng.normal(size=ATOM_DIM))
        sc = [np.abs(X) @ g for X in X_atoms]
        lifts.append(lift_from_overlap(overlap_from_scores(sc, tie_seed=30_000 + i)))
    lifts = np.array(lifts)
    rows.append({"baseline": "random feature weighting (no network)", "n_draws": 200,
                 "mean": lifts.mean(), "sd": lifts.std(ddof=1),
                 "p2_5": np.percentile(lifts, 2.5), "p97_5": np.percentile(lifts, 97.5)})

    lifts = []
    for i in range(200):
        v = rng.normal(size=FP_BITS)
        sc = shap_to_atom_scores(v, signed=True)
        lifts.append(lift_from_overlap(overlap_from_scores(sc, tie_seed=40_000 + i)))
    lifts = np.array(lifts)
    rows.append({"baseline": "random values through bit-to-atom mapping", "n_draws": 200,
                 "mean": lifts.mean(), "sd": lifts.std(ddof=1),
                 "p2_5": np.percentile(lifts, 2.5), "p97_5": np.percentile(lifts, 97.5)})

    lifts = []
    for i in range(200):
        r2 = np.random.default_rng(50_000 + i)
        sc = [r2.random(e["n_atoms"]) for e in EVAL]
        lifts.append(lift_from_overlap(overlap_from_scores(sc, tie_seed=50_000 + i)))
    lifts = np.array(lifts)
    rows.append({"baseline": "random atom ranking", "n_draws": 200,
                 "mean": lifts.mean(), "sd": lifts.std(ddof=1),
                 "p2_5": np.percentile(lifts, 2.5), "p97_5": np.percentile(lifts, 97.5)})

    base_df = pd.DataFrame(rows)
    base_df.to_csv(base_path, index=False)
print(base_df.round(4).to_string(index=False))

                                 baseline  n_draws   mean     sd   p2_5  p97_5
    random feature weighting (no network)      200 0.9890 0.1608 0.7560 1.2137
random values through bit-to-atom mapping      200 1.0097 0.1055 0.8042 1.2102
                      random atom ranking      200 1.0023 0.0205 0.9638 1.0480


## 11. Explanation stability

Integrated Gradients is deterministic given a model, while GNNExplainer optimises a mask
from a random initialisation. Both are run repeatedly on the same molecules with the same
number of repeats, and the agreement between the atom sets selected on different runs is
measured by the Jaccard index.

In [17]:
N_REPEATS = 5
stab_path = f"{OUTDIR}/table16_stability.csv"
if os.path.exists(stab_path):
    stab_df = pd.read_csv(stab_path)
else:
    def topk_sets(scores_list, k_frac=K_FRAC, tie_seed=0):
        rng = np.random.default_rng(tie_seed)
        out = []
        for e, sc in zip(EVAL, scores_list):
            if sc is None or len(sc) != e["n_atoms"]:
                out.append(None); continue
            n = e["n_atoms"]; k = max(1, int(round(k_frac * n)))
            order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
            out.append(set(order[:k].tolist()))
        return out

    def mean_pairwise_jaccard(runs):
        vals = []
        for i in range(len(EVAL)):
            sets = [r[i] for r in runs if r[i] is not None]
            if len(sets) < 2:
                continue
            js = [len(a & b) / len(a | b)
                  for x, a in enumerate(sets) for b in sets[x + 1:]]
            if js:
                vals.append(float(np.mean(js)))
        return np.array(vals)

    rows = []
    ig_runs = [topk_sets(batched_gradient_scores(gin_rep, EVAL_GRAPHS, method="ig"),
                         tie_seed=60_000 + r) for r in range(N_REPEATS)]
    j = mean_pairwise_jaccard(ig_runs)
    rows.append({"explainer": "Integrated Gradients", "n_repeats": N_REPEATS,
                 "n_molecules": len(j), "jaccard_mean": j.mean(),
                 "jaccard_sd": j.std(ddof=1)})

    gx_runs = [topk_sets(gnnexplainer_scores(gin_rep, EVAL_GRAPHS, seed=70_000 + r),
                         tie_seed=70_000 + r) for r in range(N_REPEATS)]
    j = mean_pairwise_jaccard(gx_runs)
    rows.append({"explainer": "GNNExplainer", "n_repeats": N_REPEATS,
                 "n_molecules": len(j), "jaccard_mean": j.mean(),
                 "jaccard_sd": j.std(ddof=1)})

    stab_df = pd.DataFrame(rows)
    stab_df.to_csv(stab_path, index=False)
print(stab_df.round(4).to_string(index=False))

  GNNExplainer produced attributions for 237/237 molecules
  GNNExplainer produced attributions for 237/237 molecules
  GNNExplainer produced attributions for 237/237 molecules
  GNNExplainer produced attributions for 237/237 molecules
  GNNExplainer produced attributions for 237/237 molecules
           explainer  n_repeats  n_molecules  jaccard_mean  jaccard_sd
Integrated Gradients          5          237        0.9866      0.0511
        GNNExplainer          5          237        0.6287      0.1581


## 12. Summary

In [18]:
print(f"Evaluation set: {N_EVAL} molecules, "
      f"mean chance overlap {CHANCE.mean():.4f}, ceiling {CEILING:.3f}\n")

def show(path, fmt):
    if os.path.exists(path):
        for _, r in pd.read_csv(path).iterrows():
            print(fmt(r))
    else:
        print(f"  ({os.path.basename(path)} not present)")

print("Lift over chance, trained models")
show(f"{OUTDIR}/table11_explainers.csv",
     lambda r: f"  {r.explainer:34s} {r.lift:.4f}   ({r.pct_of_ceiling:5.1f}% of ceiling)")
print("\nBaselines")
show(f"{OUTDIR}/table15_baselines.csv",
     lambda r: f"  {r.baseline:34s} {r['mean']:.4f}   [{r.p2_5:.4f}, {r.p97_5:.4f}]")
print("\nUntrained initialisations")
show(f"{OUTDIR}/table12_untrained_null.csv",
     lambda r: f"  {r.arch.upper():4s} (n={int(r['n'])})                        "
               f"{r['mean']:.4f}   [{r.p2_5:.4f}, {r.p97_5:.4f}]")
print("\nLabel-permuted controls")
show(f"{OUTDIR}/table14_label_permuted.csv",
     lambda r: f"  {r.model:34s} {r.lift:.4f}   "
               f"(train AUC {r.train_auc:.3f}, test AUC {r.test_auc:.3f})")

print("\nArtefacts written to", OUTDIR)
for f in ["table10_ranking_convention.csv", "table11_explainers.csv",
          "table12_untrained_null.csv", "table13_training_gain.csv",
          "table14_label_permuted.csv", "table15_baselines.csv",
          "table16_stability.csv"]:
    print("  -", f)

Evaluation set: 237 molecules, mean chance overlap 0.4360, ceiling 1.914

Lift over chance, trained models
  XGBoost (TreeSHAP, signed)         1.5422   ( 59.3% of ceiling)
  XGBoost (TreeSHAP, absolute)       1.4416   ( 48.3% of ceiling)
  GIN (Integrated Gradients)         1.2265   ( 24.8% of ceiling)
  GIN (Gradient x Input)             1.2164   ( 23.7% of ceiling)
  GIN (Saliency)                     1.1784   ( 19.5% of ceiling)
  GIN (GNNExplainer)                 0.8978   (-11.2% of ceiling)

Baselines
  random feature weighting (no network) 0.9890   [0.7560, 1.2137]
  random values through bit-to-atom mapping 1.0097   [0.8042, 1.2102]
  random atom ranking                1.0023   [0.9638, 1.0480]

Untrained initialisations
  GAT  (n=500)                        1.0859   [0.7976, 1.2399]
  GCN  (n=500)                        1.0370   [0.7580, 1.2450]
  GIN  (n=500)                        1.1156   [0.9894, 1.2262]

Label-permuted controls
  GIN (permuted labels)              1.1440

In [19]:
!cd /content/drive/MyDrive/ames_pipeline && zip -qr /content/part4_results.zip results
from google.colab import files
files.download("/content/part4_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>